# Tuning Imputer Strategy as Part of Grid Search

The imputation strategy (`mean` vs `median`, `most_frequent` vs `constant`) is just another hyperparameter — this notebook tunes it **together with** the model's own hyperparameters in one `GridSearchCV`, rather than picking a strategy by hand before ever touching the model.

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.head()

In [ ]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [ ]:
df.head()

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
X_train.head()

## 2. Separate Preprocessing Pipelines per Column Type

Numeric columns get median imputation + scaling; categorical columns get mode imputation + one-hot encoding. Each is its own small `Pipeline`, nested inside the bigger `ColumnTransformer` below — a `Pipeline` can contain other pipelines as steps.

In [ ]:
numerical_features = ['Age', 'Fare']
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['Embarked', 'Sex']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

In [ ]:
from sklearn import set_config

set_config(display='diagram')
clf

Evaluating `clf` alone renders this nested structure as an interactive diagram — handy for seeing the whole pipeline at a glance before tuning it.

## 3. Tuning Everything Together

The triple-underscore-free single `__` separator chains through **nested** pipeline levels: `'preprocessor__num__imputer__strategy'` means *preprocessor → the `num` sub-pipeline → its `imputer` step → its `strategy` parameter*. `GridSearchCV` tries every combination of imputer strategies *and* the classifier's regularization strength `C` together — since the best imputation choice can genuinely depend on which model/hyperparameters follow it.

In [ ]:
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__cat__imputer__strategy': ['most_frequent', 'constant'],
    'classifier__C': [0.1, 1.0, 10, 100]
}

grid_search = GridSearchCV(clf, param_grid, cv=10)

In [ ]:
grid_search.fit(X_train, y_train)

print(f"Best params:")
print(grid_search.best_params_)

In [ ]:
print(f"Internal CV score: {grid_search.best_score_:.3f}")

## 4. Inspecting Every Combination Tried

In [ ]:
import pandas as pd

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values("mean_test_score", ascending=False)
cv_results[['param_classifier__C','param_preprocessor__cat__imputer__strategy','param_preprocessor__num__imputer__strategy','mean_test_score']]

### Visualizing the Grid Search Results

With this many combinations, a heatmap of mean test score (averaged across the two categorical-imputer strategies) makes the pattern across numeric-imputer strategy vs. `C` much easier to read than scanning the table above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pivot = cv_results.pivot_table(
    values='mean_test_score',
    index='param_preprocessor__num__imputer__strategy',
    columns='param_classifier__C',
    aggfunc='mean'
)
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
plt.title('Mean CV Accuracy by Numeric Imputer Strategy and C')
plt.show()


### Using the Best Pipeline

`grid_search.best_estimator_` is the full pipeline — preprocessing *and* model — already refit with the winning combination on the entire training set. Ready to evaluate on the untouched test set, or export directly.

In [ ]:
from sklearn.metrics import accuracy_score

best_pipe = grid_search.best_estimator_
y_pred = best_pipe.predict(X_test)
print('Test accuracy:', accuracy_score(y_test, y_pred))


## Summary

| Task | Method |
|---|---|
| Nested pipeline parameter name | `'step__substep__param'` — one `__` per nesting level |
| Tune imputer strategy + model together | one `param_grid` covering both, passed to `GridSearchCV` |
| See every combination's score | `pd.DataFrame(grid_search.cv_results_)` |
| Visualize the search | pivot + `sns.heatmap()` |
| Get the fully-fitted winning pipeline | `grid_search.best_estimator_` |

Tuning imputation alongside the model avoids assuming a "best" strategy in isolation — sometimes a different imputer works better paired with a particular regularization strength, and this approach finds that combination directly instead of guessing.